In [1]:
import os
os.chdir("..")

In [2]:
from src.config import *
from src.helpers import *
from scipy.signal import resample
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [3]:
rawdata = get_rawdata()

# Preprocessing

Processed data will be saved as .csv files to my processed data directory, which contains 'chest' and 'wrist' subdirectories for the data. At each step, I will again showcase subject 2's chest data to assess the preprocessing.

## Resampling

I need to resample the signals since the chest and wrist signals are sampled at different rates. I will resample chest signals down to 64hz, and interpolate the wrist's ACC, EDA, and TEMP signals up to 64hz.

The target resampled rate is 64hz, which means the target resampled length will be 64 times the signal duration in seconds, which can be calculated by dividing the signal length by its original sampling rate. This only needs to be done for the chest signals, as these are the signals being sampled down. The non-64hz wrist signals are sampled up.

Resampling all signals to 64hz aligns their time axis, including their labels, so that the window feature extractions produce consistent input for the model.

In [4]:
def resample_signals(subnum):
    ## CHEST RESAMPLING
    chest_data = raw2df(rawdata, subnum, 'chest')
    target_length = int((chest_data.shape[0] / CHEST_HZ)*64)

    # Resample chest signals, except labels (labels are read in with chest signals)
    chest_resampled = {}
    for col in chest_data.columns:
        if col != "label":
            chest_resampled[col] = resample(chest_data[col].values, target_length)

    
    # LABEL RESAMPLING
    labels_resampled = resample(chest_data["label"].values, target_length)
    labels_resampled = np.round(labels_resampled).astype(int) #have to round
    chest_resampled["label"] = labels_resampled #add labels to resampled chest resampled
    
    chest_resampled_df = pd.DataFrame(chest_resampled) #chest and labels to df

    
    ## WRIST RESAMPLING
    wrist_data = raw2df(rawdata, subnum, 'wrist')
    wrist_resampled = {}

    # Wrist data is a dictionary of dataframes
    for sig_name, sig_data in wrist_data.items():
        sig = sig_data.iloc[:,0].values #get signal values
        if sig_name in ["EDA", "TEMP"]:
            #interpolate 4hz up to 64hz
            wrist_resampled[sig_name] = np.interp(np.linspace(0, len(sig)-1, target_length), np.arange(len(sig)), sig)
        elif sig_name.startswith("ACC"):
            #interpolate 32hz up to 64hz
            wrist_resampled[sig_name] = resample(sig, target_length)
        else:
            # if already 64hz (BVP)
            if len(sig) == target_length:
                wrist_resampled[sig_name] = sig
            else:
                wrist_resampled[sig_name] = resample(sig, target_length)

    wrist_resampled_df = pd.DataFrame(wrist_resampled) #wrist to df

    return chest_resampled_df, wrist_resampled_df

# resample subject 2 for showcase
s2chest, s2wrist = resample_signals(2)

In [5]:
print(f"There are now {len(s2chest)} samples from each subject, for each device.")

There are now 389056 samples from each subject, for each device.


## Saving processed data

Here I save the preprocessed data to my processed data directory for retrieval later come training. 

In [6]:
# function to save the processed data accordingly
def save_processed(subnum, chest_df, wrist_df):
    chest_path = os.path.join(PRODIR, "chest", f"s{subnum}chest64.csv")
    wrist_path = os.path.join(PRODIR , "wrist", f"s{subnum}wrist64.csv")

    chest_df.to_csv(chest_path, index=False)
    wrist_df.to_csv(wrist_path, index=False)

    print(f"Subject {subnum} chest and wrist data saved.")

In [7]:
# preprocess and save every subject's data
for subnum in sorted(rawdata.keys()):
    sXchest_rs, sXwrist_rs = resample_signals(subnum) #resample both
    save_processed(subnum, sXchest_rs, sXwrist_rs) #save

Subject 2 chest and wrist data saved.
Subject 3 chest and wrist data saved.
Subject 4 chest and wrist data saved.
Subject 5 chest and wrist data saved.
Subject 6 chest and wrist data saved.
Subject 7 chest and wrist data saved.
Subject 8 chest and wrist data saved.
Subject 9 chest and wrist data saved.
Subject 10 chest and wrist data saved.
Subject 11 chest and wrist data saved.
Subject 13 chest and wrist data saved.
Subject 14 chest and wrist data saved.
Subject 15 chest and wrist data saved.
Subject 16 chest and wrist data saved.
Subject 17 chest and wrist data saved.


## Windowing 
Each subject now has 389,055 data points. I will window the signal into 10 second timeframes with 5 second overlap between each window. I will use a 50% threshold to determine the majority label in each window, else the window will be ignored. That is, if the majority label isn't 50% or more of the window, the window is undefined. This is done from the window_data_features() function from src.helpers. This function also returns the summary staistics of the window instead of the raw data itself. This is done to save on memory and speed up training.

The summary statistics extracted for each channel include mean, standard deviation, minimum, maximum, and peak-to-peak amplitude. Each window is represented as a single feature vector containing these statistics across all the signals from both the chest and wrist devices. This approach allows the Random Forest to learn from informative window-level features without being overwhelmed by the full raw signals. It also ensures that the features are consistent across windows and subjects, which is  important for the LOSO cross-validation.

In [10]:
# to show the shape of a window
X2, y2 = window_data_features(subnum=2)
print(X2.shape)
print(y2.shape)

(1214, 70)
(1214,)
